In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
from typing import TypedDict

from openai import OpenAI
from pydantic import BaseModel



In [ ]:
from unittest import mock
from unittest.mock import Mock
from fastcore.test import *
from unittest.mock import MagicMock
from unittest.mock import patch



## Using an LLM to verify

The NER model approach above is quite fast and often accurate, even on a personal use computer and CPU. Nevertheless, it does make mistakes, so manual correction is needed. The following is an LLM process that attempts to find if a the NER model has made mistakes when marking definitions and notations on a note and thus the note needs manual correction.

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
DEF_NOTAT_VERIFY_SYSTEM_PROMPT = r"""
You are an expert auditor of mathematical texts. Your task is to validate semantic HTML markings (attributes: "definition" or "notation") within an excerpt. You must determine if the current markings correctly identify **newly introduced** terms while ignoring **contextual** objects.

### I. The Core Binary Classification
For every mathematical object or term, apply this binary test.

**1. THE TARGET (Must be Marked)**
A term is a Target if and only if the excerpt **establishes its meaning for the first time** intended for use beyond the immediate sentence.
*   **New Constructions:** "For any field $K$, let <span notation>$G_K$</span> denote..." (Mark $G_K$).
*   **Formal Definitions:** "We say a sheaf is <b definition>flasque</b> if..." (Mark "flasque").
*   **Explicit Assignments:** "We define the <b definition>L-series</b> <span notation>$L(s, \chi)$</span> as..." (Mark "L-series" and $L(s, \chi)$).
*   **Self-Contained "Recalls":** If the text says "Recall that X is called Y if [Definition]", and the excerpt contains the actual definition criteria, Mark Y. The author is establishing the definition for this text.
    *   *Correct:* "Recall that $X$ is <b definition>totally disconnected</b> if connected components are points."
    *   *Incorrect (Don't mark):* "Recall the properties of totally disconnected spaces."

**2. THE CONTEXT (Must NOT be Marked)**
Everything else is Context. This includes:
*   **Generic Variables:** "Let $X$ be a scheme..." ($X$ is a generic placeholder).
*   **Specific Instances/Applications:** "Let $G = \text{Gal}(L/K)$..." ($G$ is just a shorthand for this specific proof).
*   **Reminders/Recalls:** "Recall that $H^i$ denotes cohomology..." (The definition exists outside this excerpt).
*   **Process Variables:** "By induction on $n$...", "It suffices to treat...", "Let $f: X \to Y$ be a morphism..."

### II. Auditing Rules

1.  **The "Defined Here" Rule:** Only mark objects if the text explicitly links the symbol/term to its formal name or construction *in this specific excerpt*. If the text implies the reader should already know it (e.g., "Consider the trace defined above"), do not mark it.
2.  **Construction vs. Instance:**
    *   "Let $f$ be the map defined in Eq 1" $\to$ **Context** (Reference to past).
    *   "Let $f$ denote the map..." $\to$ **Target** (Establishing new notation).
3.  **OCR Robustness:** Treat malformed LaTeX (e.g., missing `$`) as valid text. If `L(s)` is a Target but lacks delimiters, it still requires a mark.

### III. Reference Examples

*   **Correct Definition:** "We call a functor $F$ <b definition>representable</b> if..."
    *   *Reason:* Defines the property "representable". $F$ is Context.
*   **Correct Notation:** "Let <span notation>$\mathbb{A}^n</span>$ denote affine space."
    *   *Reason:* Explicit assignment of global notation.
*   **False Positive (Do not mark):** "Let $C$ be a category. If $C$ has limits..."
    *   *Reason:* $C$ is a generic variable used to set the stage.
*   **False Positive (Do not mark):** "The fiber product $X \times_Y Z$ (see Chapter 1)..."
    *   *Reason:* Reference to a prior definition.

### IV. Output Format
Return a JSON object with:
1. has_incorrect_markings: Set to true if the text contains a <b definition> or <span notation> that violates the Context rules (i.e., a False Positive).
2. has_missing_markings: Set to true if the text contains a term that meets the Target rules but has no HTML tags (i.e., a False Negative).
3. "reasoning": string (Strictly limited to 30 words).

### V. Constraints on Reasoning
- DO NOT provide an introductory "Step-by-step" or "Let's analyze" paragraph.
- DO NOT repeat the rules or definitions of Target/Context.
- DO NOT provide a preamble. Start immediately with the analysis.
- DO NOT quote the excerpt.
- FORMAT: Use a simple list: [Term]: [Target/Context] - [3-word reason].
- If no errors are found, the reasoning should simply be "All markings conform to rules."

Output Requirement: You must start the reasoning string with the character [ and follow the list format. Do not use conversational fillers.
"""

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
DEF_NOTAT_VERIFY_USER_PROMPT = r"""
Audit the following excerpt of mathematical text for definition and notation marking errors. Return the result in the specified JSON format.

"""

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
class AuditResult(BaseModel):
    reasoning: str
    has_incorrect_markings: bool
    has_missing_markings: bool

class AuditVoteResult(TypedDict, total=True):
    should_remove: bool
    should_add: bool
    incorrect_tally: int  # Added tally for incorrect markings
    missing_tally: int    # Added tally for missing markings
    error_tally: int    # New: specifically tracks failed LLM calls
    total_votes: int      # Useful for context/percentage
    stats: str

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
INITIAL_ERROR_MESSAGE = "ERROR during LLM call"
def salvage_audit_result(raw_content: str, error_msg: str) -> AuditResult:
    """
    Attempts to extract boolean flags from a truncated or malformed JSON string.
    Returns a 'Success' AuditResult if flags are found, otherwise returns an Error result.
    """
    if raw_content:
        if len(raw_content.strip()) < 10:
             return AuditResult(reasoning=f"{INITIAL_ERROR_MESSAGE}: Empty response", 
                               has_incorrect_markings=False, has_missing_markings=False)
        # Normalize: remove whitespace and newlines for robust string matching
        clean = raw_content.lower().replace(" ", "").replace("\n", "").replace("\\n", "")
        
        def extract_bool(key):
            if f'"{key}":true' in clean: return True
            if f'"{key}":false' in clean: return False
            return None

        inc = extract_bool("has_incorrect_markings")
        mis = extract_bool("has_missing_markings")

        # If we found BOTH flags, we can salvage the vote even if JSON is broken
        if inc is not None and mis is not None:
            # <--- CHANGE: Extract only the text after the flags if possible
            # This makes the "Reasonings" section of your final report readable
            display_text = raw_content.split('"reasoning":')[-1].strip(' "}').replace('\\n', '\n')
            return AuditResult(
                reasoning=f"TRUNCATED SALVAGE: {display_text[:200]}...",
                has_incorrect_markings=inc,
                has_missing_markings=mis
            )
            # return AuditResult(
            #     # reasoning=f"TRUNCATED SALVAGE: {raw_content[:150]}...",
            #     # has_incorrect_markings=inc,
            #     # has_missing_markings=mis
            # )
            
    # If flags are missing, return the error string to trigger 'ignore' in voting logic
    return AuditResult(
        reasoning=f"{INITIAL_ERROR_MESSAGE}: {error_msg}",
        has_incorrect_markings=False,
        has_missing_markings=False
    )

In [ ]:
# --- Test Case 1: Standard Truncation ---
# The model finished the flags but cut off during the reasoning text.
raw_1 = '{"has_incorrect_markings": true, "has_missing_markings": false, "reasoning": "The term...'
res_1 = salvage_audit_result(raw_1, "EOF Error")
test_eq(res_1.has_incorrect_markings, True)
test_eq(res_1.has_missing_markings, False)
test_eq(res_1.reasoning.startswith("TRUNCATED SALVAGE"), True)

# --- Test Case 2: Reversed Property Order ---
# Validates that we aren't relying on 'has_incorrect' appearing before 'has_missing'.
raw_2 = '{"has_missing_markings": true, "has_incorrect_markings": false, "reasoning": "Logic...'
res_2 = salvage_audit_result(raw_2, "EOF Error")
test_eq(res_2.has_missing_markings, True)
test_eq(res_2.has_incorrect_markings, False)

# --- Test Case 3: Failed Salvage (Missing One Flag) ---
# The model cut off before the second boolean was written.
raw_3 = '{"has_incorrect_markings": true, "has_missing_markings": '
res_3 = salvage_audit_result(raw_3, "EOF Error")
# Reasoning should start with the error message so the voting logic ignores it
test_eq(res_3.reasoning.startswith(INITIAL_ERROR_MESSAGE), True)

# --- Test Case 4: Failed Salvage (Neither Flag) ---
# The model put reasoning first (against instructions) and crashed.
raw_4 = '{"reasoning": "I think this is a target because..."'
res_4 = salvage_audit_result(raw_4, "EOF Error")
test_eq(res_4.reasoning.startswith(INITIAL_ERROR_MESSAGE), True)
test_eq(res_4.has_incorrect_markings, False) # Should return default false

# --- Test Case 5: Complex Formatting ---
# Handles whitespace, newlines, and mixed casing.
raw_5 = '{\n  "HAS_INCORRECT_MARKINGS": FALSE,\n  "has_missing_markings": TRUE,\n  "reasoning": "..."'
res_5 = salvage_audit_result(raw_5, "Validation Error")
test_eq(res_5.has_incorrect_markings, False)
test_eq(res_5.has_missing_markings, True)

In [ ]:
#| hide
def test_salvage_cleaning_logic():
    "Test that salvage correctly extracts flags even when reasoning is cut off"
    # Simulated raw content where reasoning is truncated
    raw = '{"has_incorrect_markings": true, "has_missing_markings": false, "reasoning": "The term $X$ is context'
    
    res = salvage_audit_result(raw, "Truncation Error")
    
    test_eq(res.has_incorrect_markings, True)
    test_eq(res.has_missing_markings, False)
    # Ensure the "TRUNCATED SALVAGE" prefix is present
    test_eq(res.reasoning.startswith("TRUNCATED SALVAGE"), True)
    # Ensure it didn't include the JSON keys in the reasoning string
    test_ne('"has_incorrect_markings": true' in res.reasoning, True)

test_salvage_cleaning_logic()

In [ ]:
#| hide
def test_silent_failure_prevention():
    "Test that empty or near-empty responses trigger an error result"
    # Case: Empty string
    res_empty = salvage_audit_result("", "Connection lost")
    test_eq(res_empty.reasoning.startswith(INITIAL_ERROR_MESSAGE), True)
    
    # Case: Extremely short response (less than 10 chars)
    res_short = salvage_audit_result("{}", "Incomplete JSON")
    test_eq("Empty response" in res_short.reasoning, True)

test_silent_failure_prevention()

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def run_strict_audit(
        client: OpenAI,
        text_to_check: str,
        system_prompt: str = DEF_NOTAT_VERIFY_SYSTEM_PROMPT,
        user_prompt: str = DEF_NOTAT_VERIFY_USER_PROMPT,
        temperature: float = 0.3,
        verbose: bool = False,
        max_tokens=1024, # Max tokens for 
        ) -> AuditResult:
    """
    Make a client (running on an LLM) audit the definition and notation markings
    in `text_to_check` to see if any definitions/notations should have been marked
    but were not (false negatives) or were marked but should not have been
    (false positives).

    Sends text to the model and forces a JSON response matching AuditResult.
    """
    raw_content = ""
    try:
        response = client.chat.completions.create(
            model="local-model-name",  # Name often ignored by local servers, but required
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"{user_prompt}:\n\n{text_to_check}"},
            ],
            temperature=temperature,  # Low temperature = more deterministic structure
            max_tokens=1024,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "audit_result",
                    "strict": True, # Setting this False may allow the model to be longer with its reasoning at the cost of potentially breaking json 
                    # "strict": False,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "has_incorrect_markings": {"type": "boolean"},
                            "has_missing_markings": {"type": "boolean"},
                            "reasoning": {"type": "string"},
                        },
                        "required": ["reasoning", "has_incorrect_markings", "has_missing_markings"],
                        "additionalProperties": False
                    }
                }
            }
        )

        # # 1. Get the raw string (The server GUARANTEES this is valid JSON matching schema)
        # raw_content = response.choices[0].message.content

        # 1. Get the raw string 
        message = response.choices[0].message

        # Check standard content first, fall back to reasoning_content
        raw_content = message.content or getattr(message, 'reasoning_content', "")

        # 2. Parse directly into Pydantic
        result = AuditResult.model_validate_json(raw_content)
        
        return result


    except Exception as e:
        return salvage_audit_result(raw_content, str(e))

In [ ]:
#| hide
def test_run_strict_audit_scope():
    "Test that raw_content initialization prevents UnboundLocalError on API failure"
    mock_client = MagicMock()
    # Simulate a BadRequestError before any content is generated
    mock_client.chat.completions.create.side_effect = Exception("Bad Request")
    
    # This should now return an AuditResult via salvage_audit_result 
    # instead of raising UnboundLocalError
    result = run_strict_audit(mock_client, "some text", verbose=False)
    
    test_eq(isinstance(result, AuditResult), True)
    test_eq(result.reasoning.startswith(INITIAL_ERROR_MESSAGE), True)
    test_eq("Bad Request" in result.reasoning, True)

test_run_strict_audit_scope()

In [ ]:
#| hide
def test_pydantic_schema_sync():
    "Verify that the manual dict in the prompt matches the Pydantic model"
    valid_json = '{"has_incorrect_markings": false, "has_missing_markings": true, "reasoning": "Test"}'
    # This mimics what model_validate_json does inside run_strict_audit
    try:
        obj = AuditResult.model_validate_json(valid_json)
        test_eq(obj.has_missing_markings, True)
    except Exception as e:
        assert False, f"Pydantic schema mismatch: {e}"

test_pydantic_schema_sync()

`run_audit_voting` makes the model vote against itself in a "best n+1 out of 2n+1" fashion.

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
ALL_AUDITS_FAILED_STRING = "All audit attempts failed due to system errors."
def run_audit_voting(
        client: OpenAI,
        text_to_check: str,
        iterations: int = 3,
        guarantee_success_count: bool = False, # New optional argument
        max_attempts: int = 10,                 # Safety break for the guaranteed mode
        verbose: bool = True,
        system_prompt: str = DEF_NOTAT_VERIFY_SYSTEM_PROMPT,
        user_prompt: str = DEF_NOTAT_VERIFY_USER_PROMPT,
        temperature: float = 0.7,
        ) -> AuditVoteResult:
    """
    Run `run_strict_audit` multiple times to let the LLM within `client`     
    vote against itself concerning whether a text has 
    false negatives or false positives for the marked definitions and notations
    """
    audits: list[AuditResult] = []
    total_attempts = 0
    successful_count = 0

    while True:
        audit = run_strict_audit(
            client, text_to_check, system_prompt, user_prompt, temperature, verbose)
        total_attempts += 1

        # Check if this specific audit was a success (didn't return the error message)
        is_error = audit.reasoning.startswith(INITIAL_ERROR_MESSAGE)
        if not is_error:
            successful_count += 1

        audits.append(audit)

        if verbose:
            status = "Error" if is_error else "Success"
            print(f"Attempt {total_attempts} ({status})")
            print(audit, '\n\n')
        # if verbose:
        #     print(f'Vote {i}')
        #     print(audit, '\n\n')
        # Termination conditions
        if guarantee_success_count:
            # Mode 2: Stop when we have enough successes or hit the safety wall
            if successful_count >= iterations or total_attempts >= max_attempts:
                break
        else:
            # Mode 1: Stop when we hit the fixed iteration count
            if total_attempts >= iterations:

                break
    # Filtering and Tallying
    failures = [a for a in audits if a.reasoning.startswith(INITIAL_ERROR_MESSAGE)]
    successes = [a for a in audits if not a.reasoning.startswith(INITIAL_ERROR_MESSAGE)]
    error_count = len(failures)
    total = len(audits)

    # We base the voting threshold only on successful audits
    # If using Mode 2, this will be exactly `iterations`.
    # If using Mode 1, this will be `iterations - error_count`.
    valid_vote_count = len(successes)
    
    if valid_vote_count == 0:
        return AuditVoteResult(
            should_remove=False, 
            should_add=False, 
            incorrect_tally=0,
            missing_tally=0,
            error_tally=error_count,
            total_votes=total,
            stats=ALL_AUDITS_FAILED_STRING
        )

    threshold = (valid_vote_count // 2) + 1
    inc_votes = sum(1 for audit in successes if audit.has_incorrect_markings)
    mis_votes = sum(1 for audit in successes if audit.has_missing_markings)

    reasonings = [f"[{i+1}] {audit.reasoning}" for i, audit in enumerate(audits)]

    truncation_count = sum(1 for a in failures if "json" in a.reasoning.lower() or "eof" in a.reasoning.lower())

    return AuditVoteResult(
        should_remove=inc_votes >= threshold,
        should_add=mis_votes >= threshold,
        incorrect_tally=inc_votes,
        missing_tally=mis_votes,
        error_tally=error_count,
        total_votes=total,
        stats=(
            f"Summary: {inc_votes} inc, {mis_votes} mis, {error_count} failed out of {total} total attempts.\n"
            f"({truncation_count} were JSON/EOF errors) out of {total} total attempts.\n"
            f"Threshold for action (based on {valid_vote_count} successes): {threshold}\n\n"
            f"Reasonings:\n" + "\n\n".join(reasonings)
        )
    )

In [ ]:

# Mock client setup - JSON response for Pydantic parsing
mock_client = Mock()
mock_response = Mock(
    model="deepseek-r1-distill-qwen-7b",
    choices=[Mock(
        message=Mock(
            content='{"reasoning": "Mock reasoning: no issues detected", "has_incorrect_markings": false, "has_missing_markings": false}'
        )
    )]
)
mock_client.chat.completions.create.return_value = mock_response

raw_text_example = r"""
Let $L/K$ be a normal separable algebraic field extension. Its Galois group $\operatorname{Gal}(L/K)$ is defined as ...
"""

# PASS THE MOCK CLIENT
print("Running Audit...")
# with patch('__main__.DEF_NOTAT_VERIFY_SYSTEM_PROMPT', "Mock audit prompt."):
    # ... mock_client setup same as above
result = run_audit_voting(client=mock_client, text_to_check=raw_text_example)


if result:
    print("\nSUCCESS:")
    print(result)
else:
    print("\nFailed to get a result.")

Running Audit...
Attempt 1 (Success)
reasoning='Mock reasoning: no issues detected' has_incorrect_markings=False has_missing_markings=False 


Attempt 2 (Success)
reasoning='Mock reasoning: no issues detected' has_incorrect_markings=False has_missing_markings=False 


Attempt 3 (Success)
reasoning='Mock reasoning: no issues detected' has_incorrect_markings=False has_missing_markings=False 



SUCCESS:
{'should_remove': False, 'should_add': False, 'incorrect_tally': 0, 'missing_tally': 0, 'error_tally': 0, 'total_votes': 3, 'stats': 'Summary: 0 inc, 0 mis, 0 failed out of 3 total attempts.\n(0 were JSON/EOF errors) out of 3 total attempts.\nThreshold for action (based on 3 successes): 2\n\nReasonings:\n[1] Mock reasoning: no issues detected\n\n[2] Mock reasoning: no issues detected\n\n[3] Mock reasoning: no issues detected'}


In [ ]:
#| notest

if False:
    client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")


    raw_text_example = r"""
    In [Kal ], Katz examined a more general class of trigonometric sums, for which he was also able to give similar uniform estimate. To explain this result, let

    $$  h: X \longrightarrow \mathbb{A}_\mathbb{Z}  $$

    be a finitely generated scheme over the affine line $\mathbb{Z}_{\mathbb{Z}}$ above $\mathbb{Z}$. For every prime $p$, every power $q$ of it, and every non-trivial character $\psi$ of $F_{q}$, one defines the following generalized Trigonometric Sum

    $$  \sum_{x \in X\left(\mathbb{F}_{q}\right)} \psi(h(x))  $$

    Let $N$ be the supremum of the dimensions of the fibers of the complexified morphism

    $$  h \otimes \mathbb{C}: X \otimes \mathbb{C} \longrightarrow \mathbb{A}_{\mathbb{C}}  $$
    """

    print("Running Audit...")
    result = run_audit_voting(client, raw_text_example)

    if result:
        print("\nSUCCESS:")
        print(result)
    #     print(f"Reasoning: {result.reasoning}")
    #     print(f"Incorrect Markings: {result.has_incorrect_markings}")
    #     print(f"Missing Markings:   {result.has_missing_markings}")
    else:
        print("\nFailed to get a result.")

In [ ]:

# client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")

In [ ]:



# raw_text_example = r"""

# 3.4.0. Let $\mathrm{F}$ be an object of $\mathrm{C}$ We denote by $\mathrm{C} / \mathrm{F}$ the following category: The objects of $\mathrm{C} / \mathrm{F}$ are the pairs formed by an object $\mathrm{X}$ of $\mathrm{C}$ and of a morphism $u$ of $\mathrm{X}$ in $\mathrm{F}$. Let $(\mathrm{X}, u)$ and $(\mathrm{Y}, v)$ two objects. A morphism from $(\mathrm{X}, u)$ to $(\mathrm{Y}, v)$ is a $g$ morphism from X to Y such that the following diagram is commutative:

# ![[Pasted image 20250209192841.png]]
# """

# print("Running Audit...")
# result = run_audit_voting(client, raw_text_example)

# if result:
#     print("\nSUCCESS:")
#     print(result)
# #     print(f"Reasoning: {result.reasoning}")
# #     print(f"Incorrect Markings: {result.has_incorrect_markings}")
# #     print(f"Missing Markings:   {result.has_missing_markings}")
# else:
#     print("\nFailed to get a result.")

`run_audit_voting_maker` uses the `MAKER` philosophy to vote --- it only stops when the majority has a relative lead over the minority.

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def run_audit_voting_maker(
        client: OpenAI,
        text_to_check: str,
        K: int = 3,
        max_samples: int = 15,
        verbose: bool = True,
        system_prompt: str = DEF_NOTAT_VERIFY_SYSTEM_PROMPT,
        user_prompt: str = DEF_NOTAT_VERIFY_USER_PROMPT,
        temperature: float = 0.7,
        callback: callable = None, # <--- NEW: Accepts a function
        ) -> AuditVoteResult:
    
    def log(msg):
        if verbose: print(msg)
        if callback: callback(msg)

    audits: list[AuditResult] = []
    
    # Leads for independent categories
    inc_diff = 0 
    mis_diff = 0
    
    # Status flags to stop updating a category once it reaches K
    inc_settled = False
    mis_settled = False

    inc_votes = 0
    mis_votes = 0
    successful_count = 0
    total_attempts = 0

    if verbose or callback:
        log(f"--- Starting Independent MAKER Audit (K={K}) ---")
        log("LEGEND:")
        log(f"  INC_LEAD: Net difference for 'Incorrect Markings' (+ means 'is incorrect and needs manual checking', - means 'is correct')")
        log(f"  MIS_LEAD: Net difference for 'Missing Markings'   (+ means 'is missing and needs manual checking',   - means 'is not missing')")
        log(f"  Target: Lead must reach +{K} or -{K} to settle.")
        log("---")

    while total_attempts < max_samples:
        # Check if both types have reached confidence
        if inc_settled and mis_settled:
            break

        audit = run_strict_audit(
            client, text_to_check, system_prompt, user_prompt, temperature, verbose)
        total_attempts += 1

        if audit.reasoning.startswith(INITIAL_ERROR_MESSAGE):
            audits.append(audit)
            continue 

        successful_count += 1
        audits.append(audit)

        # Update Incorrect Markings Tally ONLY if not yet settled
        if not inc_settled:
            if audit.has_incorrect_markings:
                inc_votes += 1
                inc_diff += 1
            else:
                inc_diff -= 1
            
            if abs(inc_diff) >= K:
                inc_settled = True
                if verbose or callback: log(f"-> Incorrect Markings SETTLED at lead {inc_diff}")

        # Update Missing Markings Tally ONLY if not yet settled
        if not mis_settled:
            if audit.has_missing_markings:
                mis_votes += 1
                mis_diff += 1
            else:
                mis_diff -= 1
            
            if abs(mis_diff) >= K:
                mis_settled = True
                if verbose or callback: log(f"-> Missing Markings SETTLED at lead {mis_diff}")

        if verbose or callback:
            log(f"Sample {total_attempts}: INC_LEAD={inc_diff} (S:{inc_settled}), MIS_LEAD={mis_diff} (S:{mis_settled})")

    # Determine final results based on locked leads
    should_remove = inc_diff >= K
    should_add = mis_diff >= K

    error_count = total_attempts - successful_count
    reasonings = [f"[{i+1}] {a.reasoning}" for i, a in enumerate(audits)]

    return AuditVoteResult(
        should_remove=should_remove,
        should_add=should_add,
        incorrect_tally=inc_votes,
        missing_tally=mis_votes,
        error_tally=error_count,
        total_votes=total_attempts,
        stats=(
            f"MAKER Result: K={K} reached in {total_attempts} attempts.\n"
            f"Final Leads: Incorrect={inc_diff}, Missing={mis_diff}\n"
            f"Settled: Inc={inc_settled}, Mis={mis_settled}\n"
            f"Reasonings:\n" + "\n\n".join(reasonings)
        )
    )

In [ ]:
#| hide
# Setup: Dummy client and result container
mock_client = mock.MagicMock()

class MockAudit:
    def __init__(self, inc, mis, reason="Reason"):
        self.has_incorrect_markings = inc
        self.has_missing_markings = mis
        self.reasoning = reason

# -------------------------------------------------------------------
# 1. Test: Independent Freezing (Locked Tally)
# -------------------------------------------------------------------
# Scenario: 'Missing' settles at sample 3. 'Incorrect' oscillates and 
# takes longer. Even if 'Missing' flips later, it should stay SETTLED.
with patch('__main__.run_strict_audit') as mock_audit_func:
    mock_audit_func.side_effect = [
        MockAudit(True,  False), # Sample 1: INC=1,  MIS=-1
        MockAudit(False, False), # Sample 2: INC=0,  MIS=-2
        MockAudit(True,  False), # Sample 3: INC=1,  MIS=-3 (MIS SETTLES FALSE)
        MockAudit(True,  True),  # Sample 4: INC=2,  MIS=-3 (MIS frozen, INC continues)
        MockAudit(True,  True)   # Sample 5: INC=3,  MIS=-3 (INC SETTLES TRUE) -> BREAK
    ]
    
    res = run_audit_voting_maker(mock_client, "text", K=3, verbose=False)
    
    # Logic Checks
    val = getattr(res, 'total_votes', res.get('total_votes'))
    test_eq(val, 5) # Must wait for BOTH to settle
    
    # Tally Checks (using dict or attribute access)
    should_add = getattr(res, 'should_add', res.get('should_add'))
    should_remove = getattr(res, 'should_remove', res.get('should_remove'))
    
    test_eq(should_add, False)    # Settled at -3 lead
    test_eq(should_remove, True)  # Settled at +3 lead

# -------------------------------------------------------------------
# 2. Test: Termination at Max Samples (Inconclusive)
# -------------------------------------------------------------------
# Scenario: The model is inconsistent. One settles, but the other 
# never reaches K before max_samples=4.
with patch('__main__.run_strict_audit') as mock_audit_func:
    mock_audit_func.side_effect = [
        MockAudit(True,  True), # 1, 1
        MockAudit(True,  True), # 2, 2
        MockAudit(True,  True), # 3, 3 (BOTH SETTLE)
        MockAudit(False, False) # Should NOT reach this
    ]
    # We test that it stops early if both settle
    res = run_audit_voting_maker(mock_client, "text", K=3, max_samples=10, verbose=False)
    test_eq(getattr(res, 'total_votes', res.get('total_votes')), 3)

# Scenario: Never settles
with patch('__main__.run_strict_audit') as mock_audit_func:
    # Always oscillates [True, False, True, False...]
    mock_audit_func.side_effect = [MockAudit(i%2==0, False) for i in range(10)]
    
    res = run_audit_voting_maker(mock_client, "text", K=3, max_samples=4, verbose=False)
    test_eq(getattr(res, 'total_votes', res.get('total_votes')), 4) # Hits cap
    test_eq(getattr(res, 'should_remove', res.get('should_remove')), False) # Not >= K